In [1]:
# ============================================================
# Update phase_separated metadata for all completed V2 runs
# ============================================================
import importlib

import md_Helpers.Create_Lattices as cl
import md_Helpers.Simulation_Helpers as sh
import md_Helpers.viz_helpers as vh
import md_Helpers.Logging_Helpers as lh

importlib.reload(cl)
importlib.reload(sh)
importlib.reload(vh)
importlib.reload(lh)


<module 'md_Helpers.Logging_Helpers' from '/home/pnichols/MDsims/md_Helpers/Logging_Helpers.py'>

In [2]:
import importlib

import md_Helpers.Logging_Helpers as lh

importlib.reload(lh)

# Recompute phase separation metadata for all completed V2 logs.
# This does NOT rerun simulations.
# It only opens saved .gsd files and updates the .hdf5 metadata.
update_report = lh.update_all_v2_phase_separation_metadata(
    density_threshold=0.2,
    voxel_fraction_threshold=0.01,
    nbins=10,
    dry_run=False,
    verbose=True,
)

update_report

Updating V2 phase-separation metadata
base_folder = /exp/e961/data/MDsims-data/pnichols/Thermalized_States_v2
phase_name = randomization
number of logs found = 386
nbins = 10
density_threshold = 0.2
voxel_fraction_threshold = 0.01
dry_run = False
Processed 25/386 logs
Processed 50/386 logs
Processed 75/386 logs
Processed 100/386 logs
Processed 125/386 logs
Processed 150/386 logs
Processed 175/386 logs
Processed 200/386 logs
Processed 225/386 logs
Processed 250/386 logs
Processed 275/386 logs
Processed 300/386 logs
Processed 325/386 logs
Processed 350/386 logs
Processed 375/386 logs

Update summary
status
updated    386
Name: count, dtype: int64

Changed counts
changed
False    386
Name: count, dtype: int64


,n_fcc_cells,target_rho,kT,old_phase_separated,new_phase_separated,low_density_fraction,density_threshold,voxel_fraction_threshold,nbins,status,log_path,state_path


In [3]:
# ============================================================
# Update all V2 CSVs from metadata
# ============================================================

import importlib
import md_Helpers.Sweep_Helpers as sw

importlib.reload(sw)


csv_update_report = sw.update_all_v2_summary_csvs_from_metadata(
    dry_run=False,
    backup=False,
    include_backups=False,
    verbose=True,
)

display(csv_update_report)

Finding V2 CSVs with log_path
base_folder = /exp/e961/data/MDsims-data/pnichols/Thermalized_States_v2
total CSVs found = 12
usable CSVs found = 5
include_backups = False

Usable CSVs:
/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v2/FCC/Master_Summaries/master_v2_randomization.csv
/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v2/FCC/n_cells_25/Sweep_Summaries/sweep_summary_ncells_25_kT_0.70_1.00_rho_0.50_0.80_nsteps_1000000.csv
/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v2/FCC/n_cells_30/Adaptive_Pressure_Window_Summaries/adaptive_pressure_window_ncells_30_kT_0p700_1p000_P_0p000_0p150_drho_0p005_nsteps_1000000_seed_1.csv
/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v2/FCC/n_cells_30/Master_Summaries/master_v2_ncells_30_randomization.csv
/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v2/FCC/n_cells_30/Sweep_Summaries/sweep_summary_ncells_30_kT_0.70_1.00_rho_0.50_0.80_nsteps_1000000.csv

Skipped CSV counts:
reason
backup_csv    7
Name: coun

,summary_path,n_fcc_cells,target_rho,kT,old_phase_separated,new_phase_separated,phase_sep_low_density_fraction,phase_sep_density_threshold,phase_sep_voxel_fraction_threshold,log_path
0,/exp/e961/data/MDsims-data/pnichols/Thermalize...,30.0,0.74,0.72,True,False,0.0,0.2,0.01,/exp/e961/data/MDsims-data/pnichols/Thermalize...
1,/exp/e961/data/MDsims-data/pnichols/Thermalize...,30.0,0.60,0.98,True,False,0.0,0.2,0.01,/exp/e961/data/MDsims-data/pnichols/Thermalize...
2,/exp/e961/data/MDsims-data/pnichols/Thermalize...,30.0,0.74,0.72,True,False,0.0,0.2,0.01,/exp/e961/data/MDsims-data/pnichols/Thermalize...
3,/exp/e961/data/MDsims-data/pnichols/Thermalize...,30.0,0.60,0.98,True,False,0.0,0.2,0.01,/exp/e961/data/MDsims-data/pnichols/Thermalize...


In [16]:
# ============================================================
# Show failed V2 phase-separation metadata updates
# ============================================================

import importlib
from pathlib import Path
import traceback
import numpy as np
import pandas as pd

import md_Helpers.Project_Paths as pp
import md_Helpers.Logging_Helpers as lh

importlib.reload(pp)
importlib.reload(lh)


# ============================================================
# Use same settings as the update cell
# ============================================================

density_threshold = 0.2
voxel_fraction_threshold = 0.01
nbins = 10

base_folder = pp.THERMALIZED_STATES_V2_ROOT
dry_run = True   # True = check only, do not rewrite metadata


# ============================================================
# Recheck all logs, but keep the failures
# ============================================================

log_paths = sorted(
    Path(base_folder).glob("**/randomization_log.hdf5")
)

failed_rows = []

for i, log_path in enumerate(log_paths, start=1):
    try:
        lh.write_phase_separation_metadata(
            log_path=log_path,
            state_path=None,
            nbins=nbins,
            density_threshold=density_threshold,
            voxel_fraction_threshold=voxel_fraction_threshold,
            updated_from_saved_gsd=True,
            dry_run=dry_run,
        )

    except Exception as error:
        failed_rows.append({
            "log_path": str(log_path),
            "error": repr(error),
            "traceback": traceback.format_exc(),
        })

    if i % 25 == 0:
        print(f"Checked {i}/{len(log_paths)} logs")


failed_df = pd.DataFrame(failed_rows)

print("\nFailed metadata updates")
print("=" * 70)
print("number failed =", len(failed_df))

failed_df

Checked 25/388 logs
Checked 50/388 logs
Checked 75/388 logs
Checked 100/388 logs
Checked 125/388 logs
Checked 150/388 logs
Checked 175/388 logs
Checked 200/388 logs
Checked 225/388 logs
Checked 250/388 logs
Checked 275/388 logs
Checked 300/388 logs
Checked 325/388 logs
Checked 350/388 logs
Checked 375/388 logs

Failed metadata updates
number failed = 3


,log_path,error,traceback
0,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"BlockingIOError(11, ""Unable to synchronously o...","Traceback (most recent call last):\n File ""/t..."
1,/exp/e961/data/MDsims-data/pnichols/Thermalize...,FileNotFoundError('Could not find matching GSD...,"Traceback (most recent call last):\n File ""/t..."
2,/exp/e961/data/MDsims-data/pnichols/Thermalize...,FileNotFoundError('Could not find matching GSD...,"Traceback (most recent call last):\n File ""/t..."


In [26]:
# Show the full first failed row
row = failed_df.iloc[2]

print("log_path:")
print(row["log_path"])

print("\nerror:")
print(row["error"])

print("\ntraceback:")
print(row["traceback"])

log_path:
/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v2/FCC/n_cells_30/rho_0.800/kT_0.660/nsteps_1000000/seed_1/randomization_log.hdf5

error:
FileNotFoundError('Could not find matching GSD state for log: /exp/e961/data/MDsims-data/pnichols/Thermalized_States_v2/FCC/n_cells_30/rho_0.800/kT_0.660/nsteps_1000000/seed_1/randomization_log.hdf5')

traceback:
Traceback (most recent call last):
  File "/tmp/ipykernel_40245/1547279520.py", line 42, in <module>
    lh.write_phase_separation_metadata(
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        log_path=log_path,
        ^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
        dry_run=dry_run,
        ^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/pnichols/MDsims/md_Helpers/Logging_Helpers.py", line 999, in write_phase_separation_metadata
    raise FileNotFoundError(
        f"Could not find matching GSD state for log: {log_path}"
    )
FileNotFoundError: Could not find matching GSD state for log: /exp/e961/data/MDsims-data/pnichols/Thermalize

In [10]:
# ============================================================
# Build n_cells-specific master CSV
# ============================================================

import importlib

import md_Helpers.Sweep_Helpers as sw

importlib.reload(sw)


master_df = sw.build_v2_master_csv(
    n_fcc_cells=30,
    phase_name="randomization",
    n_last=100,
    dry_run=False,
    backup=False,
    verbose=True,
)

print(sw.get_v2_master_csv_path(
    n_fcc_cells=30,
    phase_name="randomization",
))

display(master_df)

Building V2 master CSV
n_fcc_cells = 30
n_cells_folder = /exp/e961/data/MDsims-data/pnichols/Thermalized_States_v2/FCC/n_cells_30
phase_name = randomization
number of logs found = 372
output_path = /exp/e961/data/MDsims-data/pnichols/Thermalized_States_v2/FCC/n_cells_30/Master_Summaries/master_v2_ncells_30_randomization.csv
n_last = 100
dry_run = False
Processed 25/372 logs
Processed 50/372 logs
Processed 75/372 logs
Processed 100/372 logs
Processed 125/372 logs
Processed 150/372 logs
Processed 175/372 logs
Processed 200/372 logs
Processed 225/372 logs
Processed 250/372 logs
Processed 275/372 logs
Processed 300/372 logs
Processed 325/372 logs
Processed 350/372 logs

Master CSV summary
status
completed         371
failed_to_read      1
Name: count, dtype: int64

Rows: 372

Wrote:
/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v2/FCC/n_cells_30/Master_Summaries/master_v2_ncells_30_randomization.csv
/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v2/FCC/n_cells_30/Master_Su

,status,master_key,n_fcc_cells,N,target_rho,actual_rho,BoxLength,volume,fcc_cell_size,kT,...,r_on_LJ,buffer_LJ,lj_mode,lattice_type,density_mode,starting_state_path,state_path,log_path,error,traceback
0,completed,n_cells_30_rho_0.500_kT_0.700_nsteps_1000000_l...,30.0,108000.0,0.50,0.50,60.000000,216000.000000,2.000000,0.7,...,2.0,0.4,xplor,fcc,fixed_N_variable_L,/exp/e961/data/MDsims-data/pnichols/Simple_Lat...,/exp/e961/data/MDsims-data/pnichols/Thermalize...,/exp/e961/data/MDsims-data/pnichols/Thermalize...,,
1,completed,n_cells_30_rho_0.520_kT_0.700_nsteps_1000000_l...,30.0,108000.0,0.52,0.52,59.220691,207692.307692,1.974023,0.7,...,2.0,0.4,xplor,fcc,fixed_N_variable_L,/exp/e961/data/MDsims-data/pnichols/Simple_Lat...,/exp/e961/data/MDsims-data/pnichols/Thermalize...,/exp/e961/data/MDsims-data/pnichols/Thermalize...,,
2,completed,n_cells_30_rho_0.540_kT_0.700_nsteps_1000000_l...,30.0,108000.0,0.54,0.54,58.480355,200000.000000,1.949345,0.7,...,2.0,0.4,xplor,fcc,fixed_N_variable_L,/exp/e961/data/MDsims-data/pnichols/Simple_Lat...,/exp/e961/data/MDsims-data/pnichols/Thermalize...,/exp/e961/data/MDsims-data/pnichols/Thermalize...,,
3,completed,n_cells_30_rho_0.560_kT_0.700_nsteps_1000000_l...,30.0,108000.0,0.56,0.56,57.775704,192857.142857,1.925857,0.7,...,2.0,0.4,xplor,fcc,fixed_N_variable_L,/exp/e961/data/MDsims-data/pnichols/Simple_Lat...,/exp/e961/data/MDsims-data/pnichols/Thermalize...,/exp/e961/data/MDsims-data/pnichols/Thermalize...,,
4,completed,n_cells_30_rho_0.580_kT_0.700_nsteps_1000000_l...,30.0,108000.0,0.58,0.58,57.103832,186206.896552,1.903461,0.7,...,2.0,0.4,xplor,fcc,fixed_N_variable_L,/exp/e961/data/MDsims-data/pnichols/Simple_Lat...,/exp/e961/data/MDsims-data/pnichols/Thermalize...,/exp/e961/data/MDsims-data/pnichols/Thermalize...,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
367,completed,n_cells_30_rho_0.740_kT_1.000_nsteps_1000000_l...,30.0,108000.0,0.74,0.74,52.649876,145945.950449,1.754996,1.0,...,2.0,0.4,xplor,fcc,fixed_N_variable_L,/exp/e961/data/MDsims-data/pnichols/Simple_Lat...,/exp/e961/data/MDsims-data/pnichols/Thermalize...,/exp/e961/data/MDsims-data/pnichols/Thermalize...,,
368,completed,n_cells_30_rho_0.760_kT_1.000_nsteps_1000000_l...,30.0,108000.0,0.76,0.76,52.183922,142105.257025,1.739464,1.0,...,2.0,0.4,xplor,fcc,fixed_N_variable_L,/exp/e961/data/MDsims-data/pnichols/Simple_Lat...,/exp/e961/data/MDsims-data/pnichols/Thermalize...,/exp/e961/data/MDsims-data/pnichols/Thermalize...,,
369,completed,n_cells_30_rho_0.780_kT_1.000_nsteps_1000000_l...,30.0,108000.0,0.78,0.78,51.734039,138461.542717,1.724468,1.0,...,2.0,0.4,xplor,fcc,fixed_N_variable_L,/exp/e961/data/MDsims-data/pnichols/Simple_Lat...,/exp/e961/data/MDsims-data/pnichols/Thermalize...,/exp/e961/data/MDsims-data/pnichols/Thermalize...,,
370,completed,n_cells_30_rho_0.800_kT_1.000_nsteps_1000000_l...,30.0,108000.0,0.80,0.80,51.299278,134999.998887,1.709976,1.0,...,2.0,0.4,xplor,fcc,fixed_N_variable_L,/exp/e961/data/MDsims-data/pnichols/Simple_Lat...,/exp/e961/data/MDsims-data/pnichols/Thermalize...,/exp/e961/data/MDsims-data/pnichols/Thermalize...,,


In [30]:
# ============================================================
# Delete one chosen V2 state
# ============================================================

import importlib

import md_Helpers.Logging_Helpers as lh

importlib.reload(lh)


# ============================================================
# Choose the state
# ============================================================

n_fcc_cells = 30
target_rho = 0.8
kT = .66
nsteps = 1_000_000
seed = 1
phase_name = "randomization"


# ============================================================
# First do a dry run
# ============================================================

delete_report = lh.delete_v2_state_files(
    n_fcc_cells=n_fcc_cells,
    target_rho=target_rho,
    kT=kT,
    nsteps=nsteps,
    seed=seed,
    phase_name=phase_name,
    dry_run=True,
    confirm_delete=True,
)

display(delete_report)

Delete V2 state files
n_fcc_cells = 30
target_rho  = 0.8
kT          = 0.66
nsteps      = 1000000
seed        = 1
phase_name  = randomization
dry_run     = True
missing | /exp/e961/data/MDsims-data/pnichols/Thermalized_States_v2/FCC/n_cells_30/rho_0.800/kT_0.660/nsteps_1000000/seed_1/randomization.gsd
missing | /exp/e961/data/MDsims-data/pnichols/Thermalized_States_v2/FCC/n_cells_30/rho_0.800/kT_0.660/nsteps_1000000/seed_1/randomization_log.hdf5
folder_missing | /exp/e961/data/MDsims-data/pnichols/Thermalized_States_v2/FCC/n_cells_30/rho_0.800/kT_0.660/nsteps_1000000/seed_1


,file_label,path,exists_before,action
0,state_path,/exp/e961/data/MDsims-data/pnichols/Thermalize...,False,missing
1,log_path,/exp/e961/data/MDsims-data/pnichols/Thermalize...,False,missing
2,folder,/exp/e961/data/MDsims-data/pnichols/Thermalize...,False,folder_missing
